# MOHID 모델 입력용 U/V 바람 성분 변환

- **입력**: `namhae_weather_master_1hr.csv` (풍향/풍속 데이터)
- **출력**: `namhae_mohid_uv_weather.csv` (U/V 성분 추가)

---

### 변환 공식
```
u = -wind_speed × sin(θ)
v = -wind_speed × cos(θ)
```
※ θ는 radian 변환 후 사용  
※ 풍향은 '불어오는 방향' 기준이므로 음수(-) 적용

### 최종 컬럼 구조
| 컬럼명 | 설명 |
|--------|------|
| time | 시간 |
| station_id | 관측소명 |
| lat, lon | 좌표 |
| wind_dir_deg | 풍향 (0~360도) |
| wind_speed_ms | 풍속 (m/s) |
| u_wind | 동서 방향 성분 (동쪽 +, 서쪽 -) |
| v_wind | 남북 방향 성분 (북쪽 +, 남쪽 -) |

In [ ]:
# CELL 1 - 라이브러리

import pandas as pd
import numpy as np
from google.colab import files

In [ ]:
# CELL 2 - 데이터 로드
# namhae_weather_master_1hr.csv 를 코랩에 업로드한 후 실행하세요

df = pd.read_csv("namhae_weather_master_1hr.csv")
print(f"로드 완료: {len(df):,}행")
print(f"컬럼: {list(df.columns)}")

In [ ]:
# CELL 3 - U/V 변환 함수 정의

def convert_to_mohid_format(df):
    """
    풍향/풍속 데이터를 MOHID 입력을 위한 U, V 벡터 성분으로 변환합니다.
    """
    # 1. 컬럼명 변경
    df = df.rename(columns={
        'datetime':   'time',
        'obs_name':   'station_id',
        'wind_dir':   'wind_dir_deg',
        'wind_speed': 'wind_speed_ms'
    })

    # 2. 풍향(Degree) -> 라디안(Radian) 변환
    theta_rad = np.radians(df['wind_dir_deg'])

    # 3. U, V 성분 계산 (바람이 불어오는 방향 기준이므로 음수 적용)
    df['u_wind'] = -df['wind_speed_ms'] * np.sin(theta_rad)
    df['v_wind'] = -df['wind_speed_ms'] * np.cos(theta_rad)

    # 4. 최종 컬럼 구조 재배치
    final_cols = ['time','station_id','lat','lon',
                  'wind_dir_deg','wind_speed_ms','u_wind','v_wind','anomaly_flag']
    return df[final_cols]

In [ ]:
# CELL 4 - 변환 실행 및 저장

mohid_df = convert_to_mohid_format(df.copy())

mohid_df.to_csv("namhae_mohid_uv_weather.csv", index=False, encoding="utf-8-sig")
print(f"저장 완료: {len(mohid_df):,}행")
print(mohid_df.head(3).to_string(index=False))

In [ ]:
# CELL 5 - 검증
# 풍향 315도, 풍속 4.2 m/s -> u≈2.97, v≈-2.97 (가이드 예시)

test_dir = 315
test_spd = 4.2
u = -test_spd * np.sin(np.radians(test_dir))
v = -test_spd * np.cos(np.radians(test_dir))
print(f"검증 (풍향 {test_dir}도, 풍속 {test_spd} m/s)")
print(f"  u = {u:.2f}  (기대값: 2.97)")
print(f"  v = {v:.2f}  (기대값: -2.97)")

print(f"\n결측률:")
for col in ["u_wind", "v_wind"]:
    rate = mohid_df[col].isna().mean()
    print(f"  {col}: {rate:.2%}")

In [ ]:
# CELL 6 - 다운로드

files.download("namhae_mohid_uv_weather.csv")
print("다운로드 완료")